In [41]:
import sqlite3
import sys
import pandas as pd

# task 1

try:
    con = sqlite3.connect('northwind.db');
except sqlite3.Error as e:
    print(f'connection to database has failed! {e}')
    sys.exit()

In [42]:
# task 2

customers_df = pd.read_sql_query("SELECT * FROM Customers", con)
orders_df = pd.read_sql_query("SELECT * FROM Orders", con)

unique_countries = customers_df['City'].unique()

print(f'Number of unique cities {unique_countries}')

count_cities = customers_df['City'].value_counts()

print(f'{count_cities}')

count_customers = orders_df['CustomerID'].value_counts().head(5)

print(f'Top five customers based on CustomerID: {count_customers}')

Number of unique cities ['Berlin' 'México D.F.' 'London' 'Luleå' 'Mannheim' 'Strasbourg' 'Madrid'
 'Marseille' 'Tsawassen' 'Buenos Aires' 'Bern' 'Sao Paulo' 'Aachen'
 'Nantes' 'Graz' 'Lille' 'Bräcke' 'München' 'Torino' 'Lisboa' 'Barcelona'
 'Sevilla' 'Campinas' 'Eugene' 'Caracas' 'Rio de Janeiro' 'San Cristóbal'
 'Elgin' 'Cork' 'Cowes' 'Brandenburg' 'Versailles' 'Toulouse' 'Vancouver'
 'Walla Walla' 'Frankfurt a.M.' 'San Francisco' 'Barquisimeto'
 'I. de Margarita' 'Portland' 'Bergamo' 'Bruxelles' 'Montréal' 'Leipzig'
 'Anchorage' 'Köln' 'Paris' 'Salzburg' 'Cunewalde' 'Albuquerque'
 'Reggio Emilia' 'Genève' 'Stavern' 'Boise' 'Kobenhavn' 'Lander'
 'Charleroi' 'Butte' 'Münster' 'Kirkland' 'Århus' None 'Lyon' 'Reims'
 'Stuttgart' 'Oulu' 'Resende' 'Seattle' 'Helsinki' 'Warszawa']
City
London            6
México D.F.       5
Sao Paulo         4
Buenos Aires      3
Rio de Janeiro    3
                 ..
Toulouse          1
Vancouver         1
Frankfurt a.M.    1
San Francisco     1
Warszawa

In [43]:
#task 3

order_details_df = pd.read_sql_query("SELECT * FROM `Order Details`",con)
products_df = pd.read_sql_query("SELECT * FROM Products",con)

merged_df = pd.merge(order_details_df,products_df,on="ProductID")

# print(merged_df)

product_counts = merged_df.groupby('ProductName')['Quantity'].sum().sort_values(ascending=False)

# print(product_counts)

merged_df['Revenue'] = merged_df['UnitPrice_x'] * (merged_df['Quantity']).astype(float)

top_revenue_product = merged_df.groupby('ProductName')['Revenue'].sum().idxmax()

max_revenue_value = merged_df.groupby('ProductName')['Revenue'].sum().max()


print(f'counts of all the products: {product_counts}\n')
print(f'top revenue product:\t {top_revenue_product}, with revenue:\t {max_revenue_value}')

counts of all the products: ProductName
Camembert Pierrot            1577
Raclette Courdavault         1496
Gorgonzola Telino            1397
Gnocchi di nonna Alice       1263
Pavlova                      1158
                             ... 
Laughing Lumberjack Lager     184
Chocolade                     138
Gravad lax                    125
Genen Shouyu                  122
Mishi Kobe Niku                95
Name: Quantity, Length: 77, dtype: int64

top revenue product:	 Côte de Blaye, with revenue:	 149984.2


In [44]:
#task 4
full_merge = customers_df.merge(orders_df, on='CustomerID') \
                         .merge(order_details_df, on='OrderID') \
                         .merge(products_df, on='ProductID')

full_merge['TotalValue'] = full_merge['UnitPrice_x'] * full_merge['Quantity']


revenue_per_country = full_merge.groupby('Country')['TotalValue'].sum().sort_values(ascending=False)
print("Total revenue per country (top 5):")
print(revenue_per_country.head())


top_customer_revenue = full_merge.groupby('CompanyName')['TotalValue'].sum().idxmax()
max_value = full_merge.groupby('CompanyName')['TotalValue'].sum().max()
print(f"Top revenue generator: {top_customer_revenue} with total: {max_value:.2f}")


Total revenue per country (top 5):
Country
USA        263566.98
Germany    244640.63
Austria    139496.63
Brazil     114968.48
France      85498.76
Name: TotalValue, dtype: float64
Top revenue generator: QUICK-Stop with total: 117483.39


In [45]:
con.close()